# 2.9 SIMD 矩阵算子示例（Matmul 高阶 API）
本示例使用 Ascend C Matmul 高阶 API 实现矩阵乘法 $C = A \times B$。Matmul 高阶 API 会封装矩阵计算过程中的数据搬运、Cube 计算调度、基础流水同步等细节，开发者主要需要完成矩阵规格配置、tiling 生成、输入输出 Tensor 设置和结果写回。

**功能介绍**：$C = A \times B$，其中 A∈[M,K]，B∈[K,N]，C∈[M,N]。本样例参数 M=512, N=512, K=128，输入 A/B 为 half 类型 ND 格式，输出 C 为 float 类型 ND 格式。


## 算子设计
**Device 端核函数**：
- `__global__ __cube__` 声明核函数，运行在 Cube 计算单元上（纯 Cube 矩阵计算场景）。
- 创建 `Matmul` 高阶 API 对象，通过 `SetTensorA`/`SetTensorB` 设置输入矩阵，`IterateAll` 执行计算并写回结果。
- `REGIST_MATMUL_OBJ` 注册 Matmul 对象，关联 TPipe、workspace 和 tiling 参数。
- 多核并行：各核通过 `GetBlockIdx()` 计算自身负责的 M 维数据段偏移。

**Host 端 Tiling 生成**：
- 使用 `MultiCoreMatmulTiling` 生成多核 Matmul 所需的 tiling 参数。
- `SetDim(2)` 设置可用 Cube 核数为 2，tiling 接口据此生成实际使用核数 `usedCoreNum`。
- `GetLibApiWorkSpaceSize()` 获取系统 workspace 大小，Host 侧申请后作为 Kernel 参数传入。
- 以 `numBlocks = tiling.usedCoreNum` 启动核函数。


## Device 端 Kernel 实现
核函数使用 Matmul 高阶 API，核心流程为：创建 GlobalTensor → 注册 Matmul 对象 → SetOrgShape → SetTensorA/SetTensorB → IterateAll → End。


In [ ]:
!mkdir -p Sources/02.09

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment ready.")


In [ ]:
%%writefile Sources/02.09/matmul_advanced_api.asc

#include <cstdint>
#include <vector>
#include <iostream>
#include <cmath>
#include "kernel_tiling/kernel_tiling.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "acl/acl.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"

using namespace matmul;

__global__ __cube__ void matmul_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* c, __gm__ uint8_t* workspace,
    AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    AscendC::GlobalTensor<half> aGlobal;
    AscendC::GlobalTensor<half> bGlobal;
    AscendC::GlobalTensor<float> cGlobal;
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(c), tiling.M * tiling.N);

    Matmul<
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>,
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>,
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, float>> mm;

    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling);
    mm.SetOrgShape(tiling.M, tiling.N, tiling.Ka, tiling.Kb);
    mm.SetTensorA(aGlobal[GetBlockIdx() * tiling.singleCoreM * tiling.Ka], false);
    mm.SetTensorB(bGlobal[0], false);
    mm.IterateAll(cGlobal[GetBlockIdx() * tiling.singleCoreM * tiling.N]);
    mm.End();
}



**关键 API 说明**：
- `MatmulType<TPosition::GM, CubeFormat::ND, half>`：模板参数描述矩阵的位置（GM）、格式（ND）和数据类型（half/float），需与 Host 侧 `SetAType`/`SetBType`/`SetCType` 保持一致。
- `REGIST_MATMUL_OBJ`：注册 Matmul 对象，关联 TPipe、系统 workspace 和 tiling 参数。
- `SetOrgShape(M, N, Ka, Kb)`：设置原始完整矩阵形状，需在 `SetTensorA`/`SetTensorB` 之前调用。
- `SetTensorA`/`SetTensorB`：设置当前核参与计算的 A/B 矩阵起始地址，第二个参数 `false` 表示不转置。A 矩阵按 M 维分核偏移，B 矩阵所有核都从首地址读取完整 B。
- `IterateAll`：执行当前核负责的全部 Matmul 计算并写回结果。
- `End`：结束 Matmul 对象使用，释放内部计算资源。


## Host 端代码实现
Host 端负责 Tiling 生成、内存分配、数据搬入、核函数启动和结果验证。

**Tiling 生成流程**：
1. 创建 `MultiCoreMatmulTiling` 对象，`SetDim(2)` 设置可用 2 个 Cube 核。
2. `SetAType`/`SetBType`/`SetCType` 设置矩阵位置、格式和数据类型。
3. `SetOrgShape`/`SetShape` 设置矩阵形状，`EnableBias(false)` 不带 bias。
4. `SetBufferSpace(-1, -1, -1)` 使用默认 L1/L0C/UB 空间。
5. `GetTiling(tilingData)` 生成最终 tiling 结果。


In [ ]:
%%writefile -a Sources/02.09/matmul_advanced_api.asc

AscendC::tiling::TCubeTiling GenerateTiling(platform_ascendc::PlatformAscendC* ascendcPlatform)
{
    constexpr int32_t M = 512;
    constexpr int32_t N = 512;
    constexpr int32_t K = 128;

    AscendC::tiling::TCubeTiling tilingData;
    matmul_tiling::MultiCoreMatmulTiling tilingApi(*ascendcPlatform);

    tilingApi.SetDim(2);
    tilingApi.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16, false);
    tilingApi.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16, false);
    tilingApi.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT);
    tilingApi.SetOrgShape(M, N, K);
    tilingApi.SetShape(M, N, K);
    tilingApi.EnableBias(false);
    tilingApi.SetBufferSpace(-1, -1, -1);

    int64_t res = tilingApi.GetTiling(tilingData);
    if (res == -1) {
        std::cout << "gen tiling failed" << std::endl;
    }
    return tilingData;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr int32_t M = 512, N = 512, K = 128;
    size_t aSize = M * K * sizeof(uint16_t);
    size_t bSize = K * N * sizeof(uint16_t);
    size_t cSize = M * N * sizeof(float);

    std::vector<uint16_t> aHost(M * K), bHost(K * N);
    for (int i = 0; i < M * K; i++) aHost[i] = (uint16_t)((i % 10) * 0.1f);
    for (int i = 0; i < K * N; i++) bHost[i] = (uint16_t)((i % 10) * 0.1f);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    size_t systemWorkspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    auto tiling = GenerateTiling(ascendcPlatform);
    uint32_t numBlocks = tiling.usedCoreNum;

    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    uint8_t *aDev=nullptr,*bDev=nullptr,*cDev=nullptr,*wsDev=nullptr,*cHost=nullptr;
    aclrtMallocHost((void**)&cHost, cSize);
    aclrtMalloc((void**)&aDev, aSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&bDev, bSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&cDev, cSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&wsDev, systemWorkspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(aDev, aSize, aHost.data(), aSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(bDev, bSize, bHost.data(), bSize, ACL_MEMCPY_HOST_TO_DEVICE);

    matmul_custom<<<numBlocks, 0, stream>>>(aDev, bDev, cDev, wsDev, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(cHost, cSize, cDev, cSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> cOut((float*)cHost, (float*)(cHost + cSize));
    bool ok = true;
    int checkCount = std::min(10, M * N);
    for (int idx = 0; idx < checkCount; idx++) {
        int m = idx / N, n = idx % N;
        float golden = 0.0f;
        for (int k = 0; k < K; k++) {
            golden += (float)((half)aHost[m * K + k]) * (float)((half)bHost[k * N + n]);
        }
        if (std::abs(cOut[idx] - golden) > 0.01f) { ok = false; break; }
    }
    std::cout << (ok ? "[Success] verification passed." : "[Failed] verification failed!") << std::endl;

    aclrtFree(aDev); aclrtFree(bDev); aclrtFree(cDev); aclrtFree(wsDev); aclrtFreeHost(cHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}


**分核与地址偏移说明**：
- `GetBlockIdx()` 表示当前核号。tiling 结果将 M=512 按 2 个核均分，每核处理 `singleCoreM=256` 行。
- A 矩阵偏移 = `GetBlockIdx() * singleCoreM * Ka`，跳过前面核已处理的行。
- B 矩阵所有核都从 `bGlobal[0]` 读取完整 B，因为每个输出元素都需要完整 K 轴累加。
- C 矩阵写回偏移 = `GetBlockIdx() * singleCoreM * N`。


## 编译与运行
Matmul 高阶 API 依赖 tiling 库，需用 CMake 构建以链接 `tiling_api`、`platform` 等库：


In [ ]:
%%writefile Sources/02.09/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "")
find_package(ASC REQUIRED)
project(demo LANGUAGES ASC CXX)
add_executable(demo matmul_advanced_api.asc)
target_link_libraries(demo PRIVATE tiling_api register platform m dl)
target_compile_options(demo PRIVATE $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>)


In [ ]:
!rm -rf Sources/02.09/build && mkdir -p Sources/02.09/build && cd Sources/02.09/build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j


In [ ]:
!if [ -e /dev/davinci0 ]; then Sources/02.09/build/demo; else cannsim record Sources/02.09/build/demo -s Ascend950 -o Sources/02.09/sim_out; fi


## 高阶 API 与基础 API 对比
| 维度 | Matmul 高阶 API（本节） | 基础 API（LoadData/Mmad） |
|---|---|---|
| Tiling | Host 侧自动生成 `TCubeTiling` | 开发者手动计算 baseM/baseK 等 |
| 数据搬运 | API 内部自动完成 GM→L1→L0A/L0B | 手动 LoadData + SetFlag/WaitFlag |
| 计算 | `IterateAll` 一行完成 | 手动 K 维循环 + Mmad 累加 |
| 同步 | API 内部管理流水同步 | 手动 SetFlag/WaitFlag |
| Workspace | 需申请系统 workspace | 无需额外 workspace |
| 代码量 | 少（~10 行 Kernel） | 多（~50 行 Kernel） |

高阶 API 大幅减少了样板代码，适合快速开发；基础 API 控制粒度更细，适合极致性能优化。


## 课后实践
尝试修改矩阵维度（如 M=256, N=256, K=256），重新生成 tiling 并观察 `usedCoreNum` 和 `singleCoreM` 的变化。


In [ ]:
!cat ./answer/02.09_answer.txt
